# TinyStories

Declares a TinyStories run end to end: both splits as sources, a BPE
tokenizer trained on the valid split alone (a 22 MB corpus, where the train
split is 2.2 GB and the trainer is single-threaded Python), a mapped dataset
over train/valid, and one pretraining leg.

Declaring is all this does. The launcher runs the jobs; their order is the
resolution printed below.

`ROOT` is where everything is declared: the volume's mount inside the lab
container, or a folder of your own on a laptop.


In [1]:
from pathlib import Path

import lab
from artifacts.core.resolve import resolve
from config import STORAGE

ROOT = STORAGE                              # the volume, inside the lab container
# ROOT = Path(".scratch/storage").resolve()  # a folder of your own, on a laptop


In [2]:
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import (
    LoopConfig,
    LRSchedule,
    OptimizerParameters,
    TrainingParameters,
)
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import ModelParameters


HF = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
train = SourceURL(name="tinystories-train", url=f"{HF}/TinyStoriesV2-GPT4-train.txt")
valid = SourceURL(name="tinystories-valid", url=f"{HF}/TinyStoriesV2-GPT4-valid.txt")

# trained on the valid split only: the merges it finds there carry over, 
# this is technically a look-forward, but for now we are testing the pipes.

# for total tokens processed D = 327,680,000
# context length 256 and batch size 64 yiels 20k steps.

D = 327680000
context_length = 256
runs = []
for batch_size in [32, 64,128]:

    step_size = int(D / context_length / batch_size)

    RUN_ID = f"TINYSTORIES_{batch_size}batch"  # <- change this per run

    tokenizer = BPETokenizer(
        vocab_size=10000,
        special_tokens=("<|endoftext|>",),
        sources=(valid,), 
        allocated_resources=Resources(cpu=4),
    )

    dataset = MappedDataSet.from_sources(
        tokenizer=tokenizer,
        train_sources=(train,),
        valid_sources=(valid,),
    )

    model_parameters = ModelParameters(
        vocab_size=tokenizer.vocab_size,
        d_model=512,
        d_ff=1344,
        sequence_length=context_length,
        num_layers=4,
        num_heads=16,
        rope_theta=10000,
        device="cuda",  # has to agree with app.GPU; the preflight checks it
        dtype="torch.float32",  # strings parsed in the worker container
    )

    training_parameters = TrainingParameters(
        total_steps=step_size,
        batch_size=batch_size,
        max_norm=1.0,
        lr_schedule=LRSchedule(
            max_learning_rate=3e-4,
            min_learning_rate=3e-5,
            warmup_iters=200,
            cosine_cycle_iters=5000,
        ),
        optimizer="torch.optim.AdamW",
        optimizer_parameters=OptimizerParameters(
            lr=3e-4,
            betas=(0.9, 0.95),
            weight_decay=0.1,
            eps=1e-8,
        ),
        seed=0,
    )

    # pretraining = Pretraining(
    #     run_id=RUN_ID,
    #     dataset=dataset,
    #     tokenizer=tokenizer,
    #     model="models.transformer",
    #     model_parameters=model_parameters,
    #     training_parameters=training_parameters,
    #     loop_config=LoopConfig(
    #         checkpoint_every=5000,
    #         val_every=200,
    #         gpu_check_every=200,
    #     ),
    #     allocated_resources=Resources(gpu_type="A100", gpu_count=1),
    # )


    runs.append(
        Pretraining(
        run_id=RUN_ID,
        dataset=dataset,
        tokenizer=tokenizer,
        model="models.transformer",
        model_parameters=model_parameters,
        training_parameters=training_parameters,
        loop_config=LoopConfig(
            checkpoint_every=5000,
            val_every=200,
            gpu_check_every=200,
        ),
        allocated_resources=Resources(gpu_type="A100", gpu_count=1),
        )
    )

## Declare

`resolve` lists what has to exist, in the order it has to happen. One job
produces one artifact, so that list *is* the work. `lab.declare` then compares
the request against what is on disk, one row per artifact, and `commit=True`
publishes whatever is `new`.


In [ ]:
for artifact in resolve(pretraining):
    print(f"{type(artifact).__name__:16} {artifact.artifact_path}")

In [ ]:
report = lab.declare(pretraining, root=ROOT, verbose=True, commit=False)

In [3]:
# this cell would only work from local repo to declare on the volume

from local import declare_on_volume
for run in runs:
    report = declare_on_volume(run, verbose=True, commit=True)  # commit=True to publish


sources/tinystories-train                                done  (drift: 73684a0 -> f9752b6)
sources/tinystories-valid                                declared  (created)
tokenizers/bpe-10000-b46983c6f3                          declared  (created)
tokenized/bpe-10000-b46983c6f3/tinystories-train         declared  (created)
tokenized/bpe-10000-b46983c6f3/tinystories-valid         declared  (created)
mappeddatasets/mapped-2be543ce28                         declared  (created)
runs/TINYSTORIES_32batch/pretraining/0-40000-f2ef90f65a  declared  (created)

6 declared, 1 done
6 manifests written: sources/tinystories-valid, tokenizers/bpe-10000-b46983c6f3, tokenized/bpe-10000-b46983c6f3/tinystories-train, tokenized/bpe-10000-b46983c6f3/tinystories-valid, mappeddatasets/mapped-2be543ce28, runs/TINYSTORIES_32batch/pretraining/0-40000-f2ef90f65a
sources/tinystories-train                                done  (drift: 73684a0 -> f9752b6)
sources/tinystories-valid                                declared

## Check a finished leg

Once a leg's `model.pt` is on the volume, `bind(ROOT)` builds its model and
loads the saved weights onto it. One validation batch through it is the
quickest confirmation that what trained is what was declared. The leg
declares `device="cuda"`; binding in the lab, which has no GPU, loads on the
CPU instead, so the batch goes wherever the model landed.

In [ ]:
from artifacts.stages.pretraining.jobs import get_batch

leg = runs[0].bind(ROOT)
inputs, labels = get_batch(
    leg.dataset.bind(ROOT).valid_tokens,
    leg.training_parameters.batch_size,
    leg.model_parameters.sequence_length,
    leg.training_parameters.seed,
    0,  # step 0's batch, the one its first training step drew
    next(leg.live_model.parameters()).device,
)
leg.live_model(inputs).shape  # (batch_size, sequence_length, vocab_size)